In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [3]:
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [4]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [5]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

generator_llm = ChatOpenAI(model="gpt-4o-mini")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [6]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [

    SummaryExtractor(llm=generator.llm),

    EmbeddingExtractor(
        embedding_model=generator.embedding_model,
        property_name="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm),
]

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),
]

In [7]:
testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=10,
    transforms=transforms,
    query_distribution=query_distribution,
    raise_exceptions=True,
)

Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [9]:
test_df = testset.to_pandas()

if test_df.empty:
    raise RuntimeError(
        "샘플이 0개입니다. 개체 추출 결과와 페르소나 매칭을 확인하세요."
    )

print(f"생성된 샘플 수: {len(test_df)}")
test_df

생성된 샘플 수: 10


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What actions did The White House take regardin...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...","The White House, through an executive order si...",AI Ethics Compliance Officer,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,일본에서 G7 정상회의는 언제 개최되었나요?,"[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...",일본에서 G7 정상회의는 2023년 5월에 개최되었습니다.,AI Policy Advisor,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,영국 AI 안전성 정상회의 뭐였어?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,영국 AI 안전성 정상회의는 2023년 11월 1~2일 블레츨리 파크에서 열린 행사...,AI Ethics Compliance Officer,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,What was the outcome of the copyright lawsuit ...,"[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...",The U.S. District Court for Northern Californi...,AI Ethics Compliance Officer,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
4,What concerns did the FTC raise regarding AI i...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,The FTC expressed concerns about potential har...,AI Safety Researcher,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
5,What stance does Germany take regarding the re...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...","Germany, along with France and Italy, opposes ...",AI Safety Researcher,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
6,마이크로소프트는 AI 안전 기금 조성에 어떻게 참여하고 있나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 프런티어 ...,"마이크로소프트는 구글, 앤스로픽, 오픈AI와 함께 프런티어 모델 포럼에 참여하여 A...",AI Policy Advisor,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
7,What role does 하버드대 로스쿨 play in the developmen...,"[SPRi AI Brief |\n2023-12월호\n코히어, 데이터 투명성 확보를 ...",하버드대 로스쿨은 AI 기업 코히어와 함께 2023년 10월 25일에 데이터 출처 ...,AI Ethics Compliance Officer,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
8,GPT-3.5는 어떤 벤치마크 테스트에서 통이치엔원 2.0에 의해 능가되었나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,"GPT-3.5는 언어 이해 테스트(MMLU), 수학(GSM8k), 질문 답변(ARC...",AI Ethics Compliance Officer,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
9,What are the key features of 삼성 가우스?,"[SPRi AI Brief |\n2023-12월호\n삼성전자, 자체 개발 생성 AI...","삼성 가우스는 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성...",AI Safety Researcher,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [10]:
test_df.to_csv("../data/ragas_synthetic_dataset.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
